# Diet Recommendation System

A reproducible walkthrough of validated diet-plan retrieval. See README.md for setup and dataset limitations. Dietary labels and nutrition units are unverified; this is an educational demo.

In [1]:
from recommender import DietRecommender, Profile, PREFERENCES
model = DietRecommender.from_csv()
df = model.data
print("Rows:", len(df))
print("Missing values:", int(df.isna().sum().sum()))
print("Exact duplicates:", int(df.duplicated().sum()))
print("Supported condition labels:", model.conditions)

Rows: 1698
Missing values: 0
Exact duplicates: 0
Supported condition labels: ['Acne', 'Diabetes', 'Heart Disease', 'Hypertension', 'Kidney Disease', 'Weight Gain', 'Weight Loss']


## Create a profile

Use readable activity names: Lightly Active maps to 1 and Moderately Active maps to 2. Calorie targets are supplied explicitly.

In [2]:
profile = Profile(age=42, weight=103, activity="Moderately Active", preference="Vegan", conditions=("Acne",), calorie_target=2000)
plans = model.recommend(profile, calorie_tolerance=0.1)
print(plans[["Dietary Preference", "Breakfast Suggestion", "Lunch Suggestion", "Dinner Suggestion", "Snack Suggestion", "Calories", "Calorie Difference", "Matched Labels", "Score"]].to_string(index=False))

Dietary Preference                                Breakfast Suggestion                           Lunch Suggestion                  Dinner Suggestion                    Snack Suggestion  Calories  Calorie Difference Matched Labels    Score
             Vegan                           Vegan pancakes with syrup                     Lentil soup with bread          Pasta with marinara sauce                  Hummus and veggies      1898                 102                0.852573
             Vegan                           Vegan pancakes with syrup                 Avocado and chickpea salad                    Vegetable curry            Apple with almond butter      1946                  54                0.656677
             Vegan Tofu scramble with vegetables and whole wheat toast Lentil and vegetable curry with brown rice  Chickpea and vegetable pasta bake Trail mix with dried fruit and nuts      2020                  20                0.651665
             Vegan                       Oat

## Explicit nutrient constraints

These example values refer to raw dataset values, whose units remain undocumented. They are demonstrations of filtering, not recommended nutrient limits.

In [3]:
filtered = model.recommend(profile, calorie_tolerance=0.1, nutrient_max={"Sugar": 150}, nutrient_min={"Protein": 60})
print(filtered[["Calories", "Protein", "Sugar", "Score"]].to_string(index=False))

 Calories  Protein  Sugar    Score
     1898      155  126.0 0.852573
     1946      103  122.0 0.656677
     2020      120  125.0 0.651665
     2040      153  102.0 0.645737
     1829      107  106.5 0.642811


## Check category adherence

This checks software constraints, not ingredient compatibility or nutritional relevance.

In [4]:
for preference in PREFERENCES:
    sample = Profile(42, 103, "Moderately Active", preference, ("Acne",))
    result = model.recommend(sample)
    assert len(result) > 0
    assert result["Dietary Preference"].eq(preference).all()
    print(preference, "category adherence: 100% for", len(result), "returned plans")
assert plans["Calories"].between(1800, 2200).all()

Omnivore category adherence: 100% for 5 returned plans
Pescatarian category adherence: 100% for 5 returned plans
Vegan category adherence: 100% for 5 returned plans
Vegetarian category adherence: 100% for 5 returned plans


## Save and reuse

The portable JSON artifact contains data and preprocessing parameters. The recommendation code supplies the inference logic.

In [5]:
model.save("diet_recommendation_model.json")
loaded = DietRecommender.load("diet_recommendation_model.json")
import pandas as pd
pd.testing.assert_frame_equal(plans, loaded.recommend(profile, calorie_tolerance=0.1), check_dtype=False)
print("Saved model reproduces the same recommendations.")

Saved model reproduces the same recommendations.
